# Notebook 6 — Classification: predict `high_engagement`

## The question
Can we predict, from what we know about a film **before people vote**, if it will have **high engagement** (many votes)?

## Vocabulary
| Term | Meaning | In our project |
|---|---|---|
| **Target** (the answer) | What we want to predict | `high_engagement` (1 = many votes, 0 = not) |
| **Features** (the clues) | What the model is allowed to look at | runtime, year, genres, budget, synopsis words... |
| **Train set** | Films the model learns from | 80% of the films |
| **Test set** | Films hidden until the end, to check the model | 20% of the films |

## The plan
1. Create the target from `vote_count` (and justify the threshold)
2. Choose the features (**no leakage**)
3. Split train / test
4. Build a **Pipeline** (preprocessing + model in one object)
5. Train **3 models**: Logistic Regression, Random Forest, Linear SVM
6. Compare **with and without the synopsis text** (does TF-IDF help?)
7. Evaluate: Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix

In [ ]:
import os

# Move to the project root so paths like "data/processed/..." always work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay)

df = pd.read_csv("data/processed/movies_features.csv")
df["overview"] = df["overview"].fillna("")      # empty text if missing (fixed value, no leakage)
print(df.shape)

## Part 1 — Create the target `high_engagement`

We build it from `vote_count`: a film has **high engagement** if it has at least a certain number of votes.

**How to choose the threshold?** First, look at the distribution.

In [ ]:
print(df["vote_count"].describe())
print()
print(df["vote_count"].quantile([0.25, 0.50, 0.75, 0.90]))

plt.figure(figsize=(8, 4))
plt.hist(np.log1p(df["vote_count"]), bins=40, color="steelblue")
plt.xlabel("log(1 + vote_count)")
plt.title("Distribution of vote_count (log scale)")
plt.show()

**Reading it**: `vote_count` is very stretched: a few films have thousands of votes, most have far fewer. So the mean is misleading; use quantiles.

**Our choice**: *high engagement = the top 25% most voted films* (75th percentile).
- **Why not the median (50%)?** It would give 50% / 50% classes. Too easy and not realistic: real "high engagement" films are the minority.
- **Why not 90%?** Too few positive films to learn from.
- **Why 75%?** About 1 film out of 4 is positive. The classes are **unbalanced** (this is why accuracy alone is misleading and F1 matters).

We turn this into a **fixed number** (rounded). It is a business rule decided once, not something learned by the model.

In [ ]:
THRESHOLD = int(round(df["vote_count"].quantile(0.75), -2))     # rounded to the nearest hundred
print("Threshold: at least", THRESHOLD, "votes")

df["high_engagement"] = (df["vote_count"] >= THRESHOLD).astype(int)

print()
print(df["high_engagement"].value_counts())
print("Share of high engagement films:", round(df["high_engagement"].mean() * 100, 1), "%")

## Part 2 — Choose the features (avoid data leakage)

**Forbidden** columns (they contain, or are too close to, the answer):
- `vote_count` → the target is built from it
- `popularity` and `vote_average` → computed from the same votes
- `revenue` → only known after the release

**Test question for each feature**: *"Would I know this value before the film gets any votes?"*

In [ ]:
numeric_features = ["runtime", "budget", "release_year", "release_month", "movie_age",
                    "n_genres", "n_keywords", "overview_word_count"]
genre_columns = [c for c in df.columns if c.startswith("genre_")]
binary_features = ["has_budget", "is_english"] + genre_columns
categorical_features = ["main_genre", "runtime_category", "original_language"]
text_feature = "overview"

forbidden = ["vote_count", "vote_average", "popularity", "revenue", "high_engagement"]
structured_features = numeric_features + binary_features + categorical_features

for column in structured_features + [text_feature]:
    assert column in df.columns, column + " is missing from the CSV"
    assert column not in forbidden, column + " is forbidden (data leakage)!"
print("OK:", len(structured_features), "structured features + the text column, no leakage")

## Part 3 — Split train / test

- `test_size=0.2` → 20% of the films are hidden for the final check.
- `stratify=y` → both sets keep the same share of high-engagement films (about 25%). Important because the classes are unbalanced.
- `random_state=42` → same split every time (reproducible).

In [ ]:
X = df[structured_features + [text_feature]]
y = df["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "| share of 1:", round(y_train.mean(), 3))
print("Test :", X_test.shape, "| share of 1:", round(y_test.mean(), 3))

## Part 4 — A baseline: the "dumb" model

A model that **always answers 0** ("not high engagement"). Any real model must do better than this.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Baseline recall  :", round(recall_score(y_test, y_pred), 3))
print("Baseline F1      :", round(f1_score(y_test, y_pred), 3))

**Why this matters**: the dumb model gets about **75% accuracy** while finding **zero** high-engagement films (recall = 0, F1 = 0). This shows why we must not trust accuracy alone.

## Part 5 — Preprocessing with `ColumnTransformer`

Different columns need different treatment:

| Column type | Treatment | Why |
|---|---|---|
| Numeric | fill missing with the **median**, then **scale** | models can't handle NaN; scaling puts runtime (~100) and n_genres (~3) on the same scale |
| Binary (0/1) | keep as it is | already numbers |
| Categorical | **One-Hot encoding** | turns "Action" / "Drama" into 0/1 columns |
| Text (`overview`) | **TF-IDF** | turns words into numbers |

**Why inside a Pipeline?** The median, the scaling and the TF-IDF vocabulary are **learned**. In a Pipeline they are learned on the **train set only**, and then applied to the test set. That is the protection against data leakage.

In [ ]:
def make_preprocessor(use_text):
    steps = [
        ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler())]), numeric_features),
        ("binary", "passthrough", binary_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
    if use_text:
        steps.append(("text", TfidfVectorizer(stop_words="english", max_features=1000,
                                              ngram_range=(1, 2), min_df=2), text_feature))
    return ColumnTransformer(steps)

# Version A = structured only | Version B = structured + TF-IDF of the synopsis
print("Preprocessors ready")

## Part 6 — The 3 models

| Model | Idea in one sentence |
|---|---|
| **Logistic Regression** | Gives a weight to each feature and adds them up; simple and easy to interpret |
| **Random Forest** | Many decision trees vote together; captures complex relations |
| **Linear SVM** | Finds the best straight line (boundary) between the two classes |

`class_weight="balanced"` makes the models pay more attention to the minority class (high engagement), because there are fewer of them.

In [ ]:
def make_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                random_state=42, n_jobs=-1),
        "Linear SVM": LinearSVC(class_weight="balanced", max_iter=5000, random_state=42),
    }

## Part 7 — Train and evaluate (A vs B)

For each of the 3 models, we train two versions: **A** (no text) and **B** (with text).

In [ ]:
def evaluate(pipe, X_test, y_test):
    y_pred = pipe.predict(X_test)
    if hasattr(pipe, "predict_proba"):
        scores = pipe.predict_proba(X_test)[:, 1]       # probability of class 1
    else:
        scores = pipe.decision_function(X_test)         # Linear SVM has no probabilities
    return {
        "accuracy": round(accuracy_score(y_test, y_pred), 3),
        "precision": round(precision_score(y_test, y_pred), 3),
        "recall": round(recall_score(y_test, y_pred), 3),
        "f1": round(f1_score(y_test, y_pred), 3),
        "roc_auc": round(roc_auc_score(y_test, scores), 3),
    }

results = []
fitted = {}

for use_text, version in [(False, "A: structured only"), (True, "B: structured + text")]:
    for name, model in make_models().items():
        pipe = Pipeline([("prep", make_preprocessor(use_text)), ("model", model)])
        pipe.fit(X_train, y_train)

        row = {"version": version, "model": name}
        row.update(evaluate(pipe, X_test, y_test))
        results.append(row)
        fitted[(version, name)] = pipe
        print("done:", version, "-", name)

results_table = pd.DataFrame(results)
results_table

**How to read the table**:
- Compare each model with the **baseline** (F1 = 0, accuracy about 0.75).
- Compare **A vs B** for the same model: does the text improve F1 and ROC-AUC?
- Compare the 3 models: which one is best, and by how much?

**Your analysis** (write it yourself):
- Which model has the best F1? Which has the best ROC-AUC?
- Did the synopsis help? A lot, a little, or not at all? (Any of these answers is valid if you show the numbers.)
- If a score is extremely high (above 0.95), suspect data leakage and check Part 2.

### The metrics in one table

| Metric | Question it answers |
|---|---|
| **Accuracy** | What share of all predictions is correct? (misleading with unbalanced classes) |
| **Precision** | When the model says "high engagement", how often is it right? |
| **Recall** | Of all the truly high-engagement films, how many did it find? |
| **F1** | One number combining precision and recall |
| **ROC-AUC** | Can the model rank a high-engagement film above a normal one? (0.5 = random, 1 = perfect) |

## Part 8 — Confusion matrix of the best model

In [ ]:
best_row = results_table.sort_values("f1", ascending=False).iloc[0]
print("Best model by F1:", best_row["model"], "|", best_row["version"], "| F1 =", best_row["f1"])

best_pipe = fitted[(best_row["version"], best_row["model"])]

ConfusionMatrixDisplay.from_estimator(best_pipe, X_test, y_test,
                                      display_labels=["normal", "high engagement"],
                                      cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()

**How to read it**:
- Top-left: normal films correctly predicted as normal.
- Bottom-right: high-engagement films correctly found (**True Positives**).
- Top-right: normal films wrongly called high engagement (**False Positives**, false alarms).
- Bottom-left: high-engagement films we missed (**False Negatives**).

**Your analysis**: which error is more frequent? For a platform that wants to promote promising films, which error is worse?

## Part 9 — Which words push toward high or low engagement?

This answers the question "how do the words help?". With Logistic Regression + text, each word has a **weight**:
positive = pushes toward high engagement, negative = pushes toward low engagement.

In [ ]:
text_pipe = fitted[("B: structured + text", "Logistic Regression")]
feature_names = text_pipe.named_steps["prep"].get_feature_names_out()
weights = text_pipe.named_steps["model"].coef_[0]

weights_table = pd.DataFrame({"feature": feature_names, "weight": weights})
word_weights = weights_table[weights_table["feature"].str.startswith("text__")].copy()
word_weights["feature"] = word_weights["feature"].str.replace("text__", "", regex=False)
word_weights = word_weights.sort_values("weight")

print("Words pushing toward LOW engagement :", list(word_weights["feature"].head(15)))
print("Words pushing toward HIGH engagement:", list(word_weights["feature"].tail(15)))

In [ ]:
# The most important features overall (structured + words)
weights_table["abs_weight"] = weights_table["weight"].abs()
weights_table.sort_values("abs_weight", ascending=False).head(15)[["feature", "weight"]]

**Your analysis**: do the words make sense? Are the most important features the structured ones (like `movie_age`, `has_budget`) or the words?

## Part 10 — Save the best model

The pipeline contains the preprocessing **and** the model, so Streamlit can reuse it without retraining.

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump({
    "model": best_pipe,
    "threshold": THRESHOLD,
    "structured_features": structured_features,
    "text_feature": text_feature,
}, "models/best_classifier.joblib")

print("Saved: models/best_classifier.joblib")

## What to remember (for the oral)
- **Target** = `high_engagement` (top 25% by `vote_count`); the threshold is justified by the distribution, and the classes are unbalanced.
- **No leakage**: `vote_count`, `popularity`, `vote_average`, `revenue` are not features. Preprocessing is inside the Pipeline, so it is learned on the train set only.
- **Baseline**: "always 0" gets about 75% accuracy, which is why we use F1, Precision, Recall and ROC-AUC.
- **3 models** compared on the same split, with and without text.
- **Confusion matrix** shows the type of errors (false alarms vs missed films).
- Next step (7): cross-validation (is the score stable?) and GridSearchCV (can we improve the best model?).